
# PCA Eigenfaces Lab (with Fun Cross-Domain Reconstructions)

**Course:** Linear Algebra  
**Topic:** PCA via SVD; eigenfaces; projection & reconstruction  
**What you'll do:**
1. Build an eigenfaces basis from a small face image library.
2. Reconstruct a held‑out face using the top \(r\) principal components.
3. Try to reconstruct a **non‑face** image (e.g., a cup of coffee) using a face basis and discuss what happens.

> This lab mirrors the classic **eigenfaces** demonstration: compute PCA/SVD on mean‑subtracted face images, obtain an orthonormal set of **eigenfaces** (columns of \(U\)), then project any image into this subspace and reconstruct. The example of approximating a dog or a cappuccino with eigenfaces illustrates how the learned subspace captures broad, smooth structures from face data.  
> *(See the PDF you shared for the Yale Face Database example and dog/coffee reconstructions.)*



## Learning goals
By the end of this lab you will be able to:
- Construct a data matrix from images and explain why **centering** matters for PCA.
- Compute PCA via the **economy SVD** and interpret: \(X_c = U\,\Sigma\,V^T\).
- Explain why the columns of **\(U\)** are called *eigenfaces* and how they form an orthonormal basis for a low‑dimensional **face subspace**.
- Project a new image \(x\) to scores \(a = U_r^T(x-\mu)\) and reconstruct \(\hat x = \mu + U_r a\).
- Analyze **variance explained** and **reconstruction error** vs. the number of components \(r\).


In [ ]:

# %% [code] Setup
import numpy as np
import matplotlib.pyplot as plt

from skimage import data, color, transform, util, exposure

np.random.seed(0)
plt.rcParams['figure.figsize'] = (6,4)

# Helper: show a list of images in a grid
from math import ceil

def show_grid(images, titles=None, ncols=6, cmap='gray', vmin=0, vmax=1, suptitle=None):
    n = len(images)
    nrows = int(ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2*ncols, 2*nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, ax in enumerate(axes):
        ax.axis('off')
        if i < n:
            ax.imshow(images[i], cmap=cmap, vmin=vmin, vmax=vmax)
            if titles is not None:
                ax.set_title(titles[i], fontsize=8)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()



## 1) Build a small *faces* dataset (no internet required)
We will synthesize a training set using two face‑like images available in `skimage`:
- **`data.camera()`** (the classic cameraman portrait — grayscale)
- **`data.astronaut()`** (convert to grayscale)

From each base image we generate multiple variants (small rotations, shifts, brightness/gamma changes, and noise) to emulate changes in pose and illumination. This provides a face library without needing to download a dataset.

> In the original eigenfaces example (Yale Faces), images of many people under different lighting/poses are stacked as **column vectors** to form the matrix \(X\). We follow the same pipeline here with a synthetic mini‑dataset.


In [ ]:

# %% [code] Create the training set of faces by augmentation

def to_gray01(img):
    if img.ndim == 3:
        img = color.rgb2gray(img)
    img = img.astype(float)
    if img.max() > 1.0:
        img = img / 255.0
    return img

# Load base images
base1 = to_gray01(data.camera())          # ~512x512
base2 = to_gray01(data.astronaut())       # RGB -> gray

# Normalize shapes to a common size (h, w)
h, w = 128, 96
base1r = transform.resize(base1, (h, w), anti_aliasing=True)
base2r = transform.resize(base2, (h, w), anti_aliasing=True)

bases = [base1r, base2r]

# Generate augmented variants
faces = []
labels = []   # 0 for base1, 1 for base2 (acts like subject ID)

rots = np.linspace(-10, 10, 5)
shifts = [(0,0), (2,0), (-2,0), (0,2), (0,-2)]
bright_scales = [0.9, 1.0, 1.1]
gammas = [0.9, 1.1]

for subj_id, base in enumerate(bases):
    for th in rots:
        rot = transform.rotate(base, th, mode='edge')
        for dy, dx in shifts:
            shifted = np.roll(np.roll(rot, dy, axis=0), dx, axis=1)
            for s in bright_scales:
                img = np.clip(shifted * s, 0, 1)
                for g in gammas:
                    img2 = exposure.adjust_gamma(img, g)
                    # small noise
                    img2 = np.clip(img2 + 0.01*np.random.randn(*img2.shape), 0, 1)
                    faces.append(img2)
                    labels.append(subj_id)

faces = np.array(faces)
labels = np.array(labels)
print(f"Training faces shape: {faces.shape}  (n_samples, h, w)")
show_grid(faces[:24], ncols=8, suptitle='A few training images')



## 2) Vectorize, center, and compute the economy SVD
Let \(x_i \in \mathbb{R}^d\) be each **vectorized** image (here, \(d=h	imes w\)). Stack them as columns:
\[ X = [x_1\; x_2\; \cdots\; x_n] \in \mathbb{R}^{d	imes n}. \]
We subtract the **mean face** \(\mu\) from each column to get \(X_c\), then compute the economy SVD:  
\[ X_c = U\,\Sigma\,V^T. \]
- Columns of **\(U\)** (size \(d	imes r\)) are the **eigenfaces**.  
- **\(\Sigma\)** holds singular values \(\sigma_1\ge\cdots\ge\sigma_r\).  
- **Explained variance** is proportional to \(\sigma_k^2\).


In [ ]:

# %% [code] PCA via SVD
n, H, W = faces.shape
D = H*W
X = faces.reshape(n, -1).T   # d × n (columns are images)
mu = X.mean(axis=1, keepdims=True)
Xc = X - mu

# Economy SVD
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)  # U: d×r, S: r, Vt: r×n
r = U.shape[1]

# Show mean face and first eigenfaces
mean_face = mu.reshape(H, W)
first_faces = [mean_face] + [U[:,i].reshape(H,W) for i in range(12)]
show_grid(first_faces, ncols=7, suptitle='Mean face (left) and first 12 eigenfaces')

# Explained variance ratio (up to scale)
explained = (S**2) / np.sum(S**2)
plt.figure(); plt.semilogy(np.arange(1,len(S)+1), S, 'o-')
plt.xlabel('Component k'); plt.ylabel('Singular value σ_k'); plt.title('Singular values (log)')
plt.show()

plt.figure(); plt.plot(np.cumsum(explained), 'o-')
plt.axhline(0.95, color='r', ls='--')
plt.xlabel('k'); plt.ylabel('Cumulative variance explained'); plt.title('Explained variance vs k')
plt.show()



## 3) Reconstruct a held‑out face using the top \(r\) components
Given a new image \(x\):  
1. Center: \(x_c = x - \mu\).  
2. Project: \(a = U_r^T x_c\).  
3. Reconstruct: \(\hat x = \mu + U_r a\).


In [ ]:

# %% [code] Reconstruction helper

def reconstruct(img2d, U, mu, r):
    x = img2d.reshape(-1, 1)
    a = U[:, :r].T @ (x - mu)
    xhat = mu + U[:, :r] @ a
    return xhat.reshape(img2d.shape)

# Pick a held-out image (not used in training of U? -- here U used all faces; for pedagogy, we can still demo)
# To do a proper holdout, recompute U on a subset. For simplicity, we'll demo with current U.

test_face = transform.resize(to_gray01(data.astronaut()), (H, W))
rs = [5, 10, 25, 50, 100, 200]
recons = [test_face] + [reconstruct(test_face, U, mu, r) for r in rs]
show_grid(recons, titles=['Original'] + [f'r={r}' for r in rs], ncols=7,
          suptitle='Reconstructing a face with top r eigenfaces')



## 4) Cross‑domain reconstruction: coffee with eigenfaces
Now try to reconstruct a **non‑face** image (a cup of coffee) using the same face basis.  
As \(r\) grows, the reconstruction captures broad, smooth structures but struggles with textures and fine details—not surprising because the **face subspace** was optimized for *face* variance, not coffee.


In [ ]:

# %% [code] Coffee reconstruction
coffee_rgb = data.coffee()
coffee = to_gray01(coffee_rgb)
coffee = transform.resize(coffee, (H, W), anti_aliasing=True)

rs = [5, 10, 25, 50, 100, 200, 400]
recons = [coffee] + [reconstruct(coffee, U, mu, r) for r in rs]
show_grid(recons, titles=['Original'] + [f'r={r}' for r in rs], ncols=4,
          suptitle='Approximating coffee using a face eigenbasis')



## 5) Try your own image (dog, cup of coffee, etc.)
Upload an image file (e.g., `dog.jpg`) to the same folder as this notebook and run the next cell. The code will:
- read and convert to grayscale
- resize to \((128, 96)\)
- reconstruct using several \(r\) values


In [ ]:

# %% [code] Your image here
#@title Reconstruct your own image using the eigenfaces basis
#@markdown Set a filename that exists in the working directory.
filename = 'dog.jpg'  # <-- change me

try:
    from skimage import io
    img = io.imread(filename)
    img = to_gray01(img)
    img = transform.resize(img, (H, W), anti_aliasing=True)

    rs = [5, 10, 25, 50, 100, 200, 400, 800]
    recons = [img] + [reconstruct(img, U, mu, r) for r in rs if r <= U.shape[1]]
    show_grid(recons, titles=['Original'] + [f'r={r}' for r in rs if r <= U.shape[1]], ncols=4,
              suptitle=f'Approximating {filename} using eigenfaces')
except FileNotFoundError:
    print(f"File '{filename}' not found. Upload it next to the notebook and re-run.")



## 6) Bonus: Visualize projections for two subjects
Project all training images onto the first two principal components and see how samples from the two subjects cluster.


In [ ]:

# %% [code] 2D projection scatterplot
scores = (U[:, :2].T @ Xc)  # 2 × n
plt.figure(figsize=(5,4))
mask0 = labels == 0
mask1 = labels == 1
plt.scatter(scores[0,mask0], scores[1,mask0], s=12, c='tab:blue', label='Subject 0 (camera)')
plt.scatter(scores[0,mask1], scores[1,mask1], s=12, c='tab:orange', label='Subject 1 (astronaut)')
plt.axhline(0, color='k', lw=0.5); plt.axvline(0, color='k', lw=0.5)
plt.legend(); plt.title('Projection onto PC1–PC2'); plt.xlabel('PC1 score'); plt.ylabel('PC2 score')
plt.grid(True, ls=':')
plt.show()



## 7) Exercises & discussion
1. **Variance vs. reconstruction**: For \(r\in\{5,10,25,50,100,200\}\), compute the relative Frobenius reconstruction error \(\|x-\hat x\|_F/\|x\|_F\) for the face and the coffee images. Plot error vs. \(r\).
2. **Choosing \(r\)**: Find the smallest \(r\) that achieves at least **95\%** cumulative variance. How do reconstructions look at this \(r\)?
3. **Whitening (optional)**: Form \(Z = \Sigma_r^{-1} U_r^T X_c\) and check that the covariance of \(Z\) is approximately the identity.
4. **Out‑of‑domain reconstructions**: Describe visually what structures eigenfaces capture when reconstructing coffee/dog images. Why does increasing \(r\) help? Why do some textures remain poor?


In [ ]:

# %% [code] Exercise 1: reconstruction error curves (example solution)

def rel_frob(A, B):
    return np.linalg.norm(A-B) / np.linalg.norm(A)

face_img = transform.resize(to_gray01(data.astronaut()), (H, W))
coffee_img = transform.resize(to_gray01(data.coffee()), (H, W))

rs = [5, 10, 25, 50, 100, 200, 400]
face_err, coffee_err = [], []
for r in rs:
    xr_f = reconstruct(face_img, U, mu, r)
    xr_c = reconstruct(coffee_img, U, mu, r)
    face_err.append(rel_frob(face_img, xr_f))
    coffee_err.append(rel_frob(coffee_img, xr_c))

plt.figure(); plt.plot(rs, face_err, 'o-', label='face')
plt.plot(rs, coffee_err, 's--', label='coffee')
plt.xlabel('r (components)'); plt.ylabel('Relative Frobenius error'); plt.title('Reconstruction error vs r')
plt.grid(True, ls=':'); plt.legend(); plt.show()



## 8) What to hand in
- A short write‑up with: (i) screenshots of reconstructions for several \(r\), (ii) the cumulative variance plot, (iii) error‑vs‑\(r\) plot, and (iv) a discussion of cross‑domain reconstructions.
- Your notebook with any modifications.

**Notes for instructors:** You can swap in a real face dataset (e.g., Yale/ORL) if available. The PCA/SVD pipeline is unchanged; only the data loader differs.
